##SCD2 - Direct Access


In [0]:
from datetime import datetime, timedelta
from pyspark.sql.functions import current_date, lit
from delta.tables import DeltaTable

# ==========================================================
# SOURCE ADLS AUTHENTICATION
# ==========================================================

spark.conf.set(
    "fs.azure.account.auth.type.adlsdevvbsrc01.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.adlsdevvbsrc01.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.adlsdevvbsrc01.dfs.core.windows.net",
    "2c5d85d1-759d-4e41-9703-905b120f74f5"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.adlsdevvbsrc01.dfs.core.windows.net",
    "YOUR_CLIENT_SECRET"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.adlsdevvbsrc01.dfs.core.windows.net",
    "https://login.microsoftonline.com/7cab1251-58e4-4efb-8936-ae9b0cb98fce/oauth2/token"
)

# ==========================================================
# TARGET ADLS AUTHENTICATION
# ==========================================================

spark.conf.set(
    "fs.azure.account.auth.type.adlsdevvbstd01.dfs.core.windows.net",
    "OAuth"
)

spark.conf.set(
    "fs.azure.account.oauth.provider.type.adlsdevvbstd01.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.id.adlsdevvbstd01.dfs.core.windows.net",
    "2c5d85d1-759d-4e41-9703-905b120f74f5"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.secret.adlsdevvbstd01.dfs.core.windows.net",
    "YOUR_CLIENT_SECRET"
)

spark.conf.set(
    "fs.azure.account.oauth2.client.endpoint.adlsdevvbstd01.dfs.core.windows.net",
    "https://login.microsoftonline.com/7cab1251-58e4-4efb-8936-ae9b0cb98fce/oauth2/token"
)

# ==========================================================
# PROCESS DATE
# ==========================================================

# Today's file
process_date = datetime.today().strftime("%Y/%m/%d")

# If you want yesterday's file instead, use:
# process_date = (datetime.today() - timedelta(days=1)).strftime("%Y/%m/%d")

print(f"Processing Date : {process_date}")

# ==========================================================
# SOURCE FILE PATH (Direct ADLS Access)
# ==========================================================

source_path = f"abfss://output@adlsdevvbsrc01.dfs.core.windows.net/dim_scd2/{process_date}/*.parquet"

print(f"Reading File : {source_path}")

# ==========================================================
# READ EVENT TRIGGERED FILE
# ==========================================================

raw_df = (
    spark.read
         .format("parquet")
         .option("inferSchema", "true")
         .load(source_path)
)

display(raw_df)

# ==========================================================
# TARGET DELTA PATH (Direct ADLS Access)
# ==========================================================

delta_path = "abfss://output@adlsdevvbstd01.dfs.core.windows.net/payment_delta_scd2"

print(f"Target Delta Path : {delta_path}")

###ADLS MOUNT FOR SCD2

In [0]:
configs = {
  "fs.azure.account.auth.type": "OAuth",
  "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
  "fs.azure.account.oauth2.client.id": "4c73a0e8-727f-4978-a885-c24c55d9f5b3",
  "fs.azure.account.oauth2.client.secret": "secretif",
  "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/3c9a95b3-9291-4ed2-9677-13948a6e69d0/oauth2/token"
}
dbutils.fs.mount(
  source = "abfss://input@adlsdevvbsorce001.dfs.core.windows.net/",
  mount_point = "/mnt/source",
  extra_configs = configs)

---------------------------------------------------------------------------
Py4JError                                 Traceback (most recent call last)
File <command-6756497593697766>, line 8
      1 configs = {
      2   "fs.azure.account.auth.type": "OAuth",
      3   "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
   (...)
      6   "fs.azure.account.oauth2.client.endpoint": "https://login.microsoftonline.com/3c9a95b3-9291-4ed2-9677-13948a6e69d0/oauth2/token"
      7 }
----> 8 dbutils.fs.mount(
      9   source = "abfss://input@adlsdevvbsorce001.dfs.core.windows.net/",
     10   mount_point = "/mnt/source",
     11   extra_configs = configs)

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:46, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     43 @functools.wraps(f)
     44 def f_with_exception_handling(*args, **kwargs):
     45     try:
---> 46         return 

In [0]:
dbutils.fs.ls("/mnt/")

---------------------------------------------------------------------------
ExecutionError                            Traceback (most recent call last)
File <command-6756497593697776>, line 1
----> 1 dbutils.fs.ls("/mnt/")

File /databricks/python_shell/lib/dbruntime/remotefshandler/RemoteFsHandler.py:52, in prettify_exception_message.<locals>.f_with_exception_handling(*args, **kwargs)
     49 class ExecutionError(Exception):
     50     pass
---> 52 raise ExecutionError(str(e)) from None

ExecutionError: [DBFS_DISABLED] Public DBFS root is disabled. Access is denied on path: /mnt SQLSTATE: 56038

JVM stacktrace:
com.databricks.backend.daemon.data.client.DbfsUnsupportedOperationSparkException
	at com.databricks.backend.daemon.data.client.DbfsExceptionMapperImpl.withExceptionWrapping(DbfsSparkException.scala:42)
	at com.databricks.backend.daemon.data.client.DBFSV2.listStatus(DatabricksFileSystemV2.scala:204)
	at com.databricks.backend.daemon.data.client.DatabricksFileSystem.listStatus(D

In [0]:
%fs
ls '/mnt/source/csv/2025/09/06'

path,name,size,modificationTime
dbfs:/mnt/source/csv/2025/09/06/olist_order_payments_dataset.parquet,olist_order_payments_dataset.parquet,4230282,1757142124000


### Picking latest data from Source Location

In [0]:
from datetime import datetime, timedelta

# yesterday’s file (assuming you process next day)
#process_date = (datetime.today() - timedelta(days=1)).strftime("%Y/%m/%d")
process_date = datetime.today().strftime("%Y/%m/%d") #--today's file

path = f"/mnt/source/csv/{process_date}/*.parquet"


raw_df = spark.read.format("parquet") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(path)

display(raw_df)

---------------------------------------------------------------------------
UnsupportedOperationException             Traceback (most recent call last)
File <command-6756497593697769>, line 15
      7 path = f"/mnt/source/csv/{process_date}/*.parquet"
     10 raw_df = spark.read.format("parquet") \
     11     .option("header", "true") \
     12     .option("inferSchema", "true") \
     13     .load(path)
---> 15 display(raw_df)

File /databricks/python_shell/lib/dbruntime/display.py:133, in Display.display(self, input, *args, **kwargs)
    131     pass
    132 elif self._cf_helper is not None and isinstance(input, ConnectDataFrame):
--> 133     self.display_connect_table(input, **kwargs)
    134 elif isinstance(input, ConnectDataFrame):
    135     if input.isStreaming:

File /databricks/python_shell/lib/dbruntime/display.py:93, in Display.display_connect_table(self, df, **kwargs)
     88 except Exception as e:
     89     raise type(
     90         e
     91     )("IPython shell enc

###DELTA TABLE

In [0]:
    raw_df.write.format("delta").mode("overwrite").save("/mnt/curated/orderpay_scd2_delta")
    spark.sql("CREATE TABLE IF NOT EXISTS orderpay_scd2_delta USING DELTA LOCATION '/mnt/curated/orderpay_scd2_delta'")

Out[17]: DataFrame[]

###SCD2

In [0]:
from pyspark.sql.functions import current_date, current_timestamp, lit, expr
from delta.tables import DeltaTable

# ---- Step 1: Get latest file path and folder date ----
folders = dbutils.fs.ls("/mnt/source/csv/")
latest_year = max([f.name.replace('/', '') for f in folders])

folders = dbutils.fs.ls(f"/mnt/source/csv/{latest_year}/")
latest_month = max([f.name.replace('/', '') for f in folders])

folders = dbutils.fs.ls(f"/mnt/source/csv/{latest_year}/{latest_month}/")
latest_day = max([f.name.replace('/', '') for f in folders])

latest_path = f"/mnt/source/csv/{latest_year}/{latest_month}/{latest_day}/*.parquet"
print(f"Reading from: {latest_path}")

# Construct file_date from folder
file_date = f"{latest_year}-{latest_month}-{latest_day}"

raw_df = spark.read.parquet(latest_path)

new_df = (raw_df
            .withColumn("ingest_date", current_date())
            .withColumn("file_date", lit(file_date))
            .withColumn("updated_at", current_timestamp())
            .withColumn("payment_dim_id", expr("uuid()"))
            .withColumn("is_active", lit(1))
            .withColumn("start_date", current_date())
            .withColumn("end_date", lit(None).cast("date")))

delta_path = "/mnt/curated/orderpay_scd2_delta"

# ---- Step 2: Initial load with SCD2 columns if table does not exist ----
if not DeltaTable.isDeltaTable(spark, delta_path):
    (new_df
        .withColumn("payment_dim_id", expr("uuid()"))
        .withColumn("is_active", lit(1))
        .withColumn("start_date", current_date())
        .withColumn("end_date", lit(None).cast("date"))
        .write
        .format("delta")
        .mode("overwrite")
        .option("mergeSchema", "true")
        .save(delta_path))
else:
    # ---- Step 3: Add missing SCD2 columns if they don’t exist ----
    deltaTable = DeltaTable.forPath(spark, delta_path)
    existing_columns = [f.name for f in deltaTable.toDF().schema.fields]

    df = deltaTable.toDF()
    # Add SCD2 columns if missing
    if "payment_dim_id" not in existing_columns:
        df = df.withColumn("payment_dim_id", lit(None).cast("string"))
    if "is_active" not in existing_columns:
        df = df.withColumn("is_active", lit(1))
    if "start_date" not in existing_columns:
        df = df.withColumn("start_date", current_date())
    if "end_date" not in existing_columns:
        df = df.withColumn("end_date", lit(None).cast("date"))

    # Add audit columns if missing
    if "file_date" not in existing_columns:
        df = df.withColumn("file_date", lit(None).cast("string"))
    if "ingest_date" not in existing_columns:
        df = df.withColumn("ingest_date", lit(None).cast("date"))
    if "updated_at" not in existing_columns:
        df = df.withColumn("updated_at", lit(None).cast("timestamp"))

    # Overwrite table with new schema
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_path)

    # Reload table for merge
    deltaTable = DeltaTable.forPath(spark, delta_path)

    # ---- Step 4: SCD2 Merge ----
    (
        deltaTable.alias("t")
        .merge(
            new_df.alias("s"),
            "t.order_id = s.order_id AND t.payment_sequential = s.payment_sequential"
        )
        .whenMatchedUpdate(
            condition="""
                t.is_active = 1 AND (
                    t.payment_type <> s.payment_type OR
                    t.payment_installments <> s.payment_installments OR
                    t.payment_value <> s.payment_value
                )
            """,
            set={
                "is_active": "0",
                "end_date": "current_date()"
            }
        )
        .whenNotMatchedInsert(values={
            "payment_dim_id": "s.payment_dim_id",
            "order_id": "s.order_id",
            "payment_sequential": "s.payment_sequential",
            "payment_type": "s.payment_type",
            "payment_installments": "s.payment_installments",
            "payment_value": "s.payment_value",
            "ingest_date": "s.ingest_date",
            "file_date": "s.file_date",
            "updated_at": "s.updated_at",
            "is_active": "1",
            "start_date": "current_date()",
            "end_date": "null"
        })
        .execute()
    )

# ---- Step 5: Check results ----
print("Before Merge:")
raw_df.show(10, truncate=False)
updated_df = spark.read.format("delta").load(delta_path)
print("After Merge:")
updated_df.show(20, truncate=False)

Reading from: /mnt/source/csv/2025/09/06/*.parquet
Before Merge:
+--------------------------------+------------------+------------+--------------------+-------------+
order_id |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
b81ef226f3fe1789b1e8b2acac839d17|1 |credit_card |8 |99.33 |
a9810da82917af2d9aefd1278f1dcfa0|1 |credit_card |1 |24.39 |
25e8ea4e93396b6fa0d3dd708e76c1bd|1 |credit_card |1 |65.71 |
ba78997921bbcdc1373bb41e913ab953|1 |credit_card |8 |107.78 |
42fdf880ba16b47b59251dd489d4441a|1 |credit_card |2 |128.45 |
298fcdf1f73eb413e4d26d01b25bc1cd|1 |credit_card |2 |96.12 |
771ee386b001f06208a7419e4fc1bbd7|1 |credit_card |1 |81.16 |
3d7239c394a212faae122962df514ac7|1 |credit_card |3 |51.84 |
1f78449c87a54faf9e96e88ba1491fa9|1 |credit_card |6 |341.09 |
0573b5e23cbd798006520e1d5b4c6714|1 |boleto |1 |51.95 |
+--------------------------------+------------------+------------+--------------------+-------------+
only showing top 10 rows

After Merge:
+--------------------------------+------------------+------------+--------------------+-------------+---------+----------+--------+---------+-----------+----------+--------------+
order_id |payment_sequential|payment_type|payment_installments|payment_value|is_active|start_date|end_date|file_date|ingest_date|updated_at|payment_dim_id|
+--------------------------------+------------------+------------+--------------------+-------------+---------+----------+--------+---------+-----------+----------+--------------+
b81ef226f3fe1789b1e8b2acac839d17|1 |credit_card |8 |99.33 |null |null |null |null |null |null |null |
a9810da82917af2d9aefd1278f1dcfa0|1 |credit_card |1 |24.39 |null |null |null |null |null |null |null |
25e8ea4e93396b6fa0d3dd708e76c1bd|1 |credit_card |1 |65.71 |null |null |null |null |null |null |null |
ba78997921bbcdc1373bb41e913ab953|1 |credit_card |8 |107.78 |null |null |null |null |null |null |null |
42fdf880ba16b47b59251dd489d4441a|1 |credit_card |2 |128.45 |null |null |null |null |null |null |null |
298fcdf1f73eb413e4d26d01b25bc1cd|1 |credit_card |2 |96.12 |null |null |null |null |null |null |null |
771ee386b001f06208a7419e4fc1bbd7|1 |credit_card |1 |81.16 |null |null |null |null |null |null |null |
3d7239c394a212faae122962df514ac7|1 |credit_card |3 |51.84 |null |null |null |null |null |null |null |
1f78449c87a54faf9e96e88ba1491fa9|1 |credit_card |6 |341.09 |null |null |null |null |null |null |null |
0573b5e23cbd798006520e1d5b4c6714|1 |boleto |1 |51.95 |null |null |null |null |null |null |null |
d88e0d5fa41661ce03cf6cf336527646|1 |credit_card |8 |188.73 |null |null |null |null |null |null |null |
2480f727e869fdeb397244a21b721b67|1 |credit_card |1 |141.90 |null |null |null |null |null |null |null |
616105c9352a9668c38303ad44e056cd|1 |credit_card |1 |75.78 |null |null |null |null |null |null |null |
cf95215a722f3ebf29e6bbab87a29e61|1 |credit_card |5 |102.66 |null |null |null |null |null |null |null |
769214176682788a92801d8907fa1b40|1 |credit_card |4 |105.28 |null |null |null |null |null |null |null |
12e5cfe0e4716b59afb0e0f4a3bd6570|1 |credit_card |10 |157.45 |null |null |null |null |null |null |null |
61059985a6fc0ad64e95d9944caacdad|1 |credit_card |1 |132.04 |null |null |null |null |null |null |null |
79da3f5fe31ad1e454f06f95dc032ad5|1 |credit_card |1 |98.94 |null |null |null |null |null |null |null |
8ac09207f415d55acff302df7d6a895c|1 |credit_card |4 |244.15 |null |null |null |null |null |null |null |
b2349a3f20dfbeef62e7b31baa22f84b|1 |credit_card |3 |136.71 |null |null |null |null |null |null |null |
+--------------------------------+------------------+------------+--------------------+-------------+---------+----------+--------+---------+-----------+----------+--------------+
only showing top 20 rows